In [4]:
#imports

import requests
import hashlib
import pyarrow
import pandas as pd
import json
import numpy as np
import pickle
from pathlib import Path
from transformers import AutoTokenizer

# October Milestone — Task 7: Baseline row-level outputs

#### Goal:
Run the two September baselines on the approved October data. Future tasks can then compare them directly with the October transformer, using the same chunks, labels and masks.

#### The comparison this sets up:
September model on October data vs. October model on October data

#### Baselines used (both frozen, nothing is retrained or re-tuned):

DistilRoBERTa embeddings + logistic regression, with the frozen 0.20 threshold
Keyword scorecards, with the thresholds selected in Task 11

#### Data:
The train (10,841 chunks) and validation (2,873 chunks) splits from October Task 3, with 10 categories per chunk. The official test set is not touched.

#### Steps:

Load the Task 3 targets

Load frozen September models

Load chunk text and embeddings (September adn October use the same) , and check they match the Task 3 chunk IDs

Generate scores and 0/1 predictions for both baselines

Combine them into one row-level table, with one row per chunk and category

Validate the table, then save it for future tasks

#### Output:
october_task7_outputs

# Step 1: Load required project data

This step needs the September Simple Regression model, October chunks, and October Embeddings.

First we get the approved October data generated in October task 3

In [35]:
# Path to Task 3 outputs
TASK3_DIR = Path("october_task3_outputs_11_2")

# Load approved chunk-category targets
train_targets = pd.read_parquet(
    TASK3_DIR / "task3_labels_mask_train.parquet"
)

validation_targets = pd.read_parquet(
    TASK3_DIR / "task3_labels_mask_validation.parquet"
)

# Load the combined row-level target table
documents = pd.read_csv(
    TASK3_DIR / "task3_chunk_category_targets.csv"
)

Finally, we just want to verify that this data matches the input we expect

In [36]:
# Check dataset sizes and columns
print("expecting Training rows: 10841")
print("found Training rows :", len(train_targets))
print("expecting Validation rows: 2873")
print("found Validation rows:", len(validation_targets))

print("\nTraining columns:", train_targets.columns.tolist())
print("\nValidation columns:", validation_targets.columns.tolist())

# Check unique chunk IDs
print("\nUnique training chunks:", train_targets["chunk_id"].nunique())
print("Unique validation chunks:", validation_targets["chunk_id"].nunique())

# Preview the data
display(train_targets.head())
display(validation_targets.head())

expecting Training rows: 10841
found Training rows : 10841
expecting Validation rows: 2873
found Validation rows: 2873

Training columns: ['chunk_id', 'labels', 'mask']

Validation columns: ['chunk_id', 'labels', 'mask']

Unique training chunks: 10841
Unique validation chunks: 2873


,chunk_id,labels,mask
0,contract_0001__chunk_0000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
1,contract_0001__chunk_0001,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 1]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,contract_0001__chunk_0002,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,contract_0001__chunk_0003,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4,contract_0001__chunk_0004,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


,chunk_id,labels,mask
0,contract_0013__chunk_0000,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
1,contract_0013__chunk_0001,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
2,contract_0013__chunk_0002,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,contract_0013__chunk_0003,"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4,contract_0013__chunk_0004,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


# Step 2: Load September Model

Now we are importing our simple classification model in order to run our October data through it

In [37]:
# Load the frozen classifier from Simple Classification

FROZEN_DIR = Path("task12b_outputs")

with open(FROZEN_DIR / "logistic_baseline.pkl", "rb") as file:
    clf = pickle.load(file)

# Step 3: Load Chunks and Embeddings

Now we are loading the chunked text (October tasks use the same chunks as September) and September embeddings ( September chunks means Sept embeds).

In [38]:
# Chunk text for the keyword baseline, embeddings for the logistic regression baseline.
# In task12b_outputs the "test_*" files are the validation chunks
#The September embeddings in task12b_outputs/ were made with the same model (all-distilroberta-v1) for the same chunks as the October Task 3 tables, so we reuse them (and check below that the chunk IDs line up)
SEPT_DIR = Path("task12b_outputs")

september_files = {
    "train": ("train_chunks.parquet", "train_embeddings.npz"),
    "validation": ("test_chunks.parquet", "test_embeddings.npz"),
}
task3_targets = {"train": train_targets, "validation": validation_targets}

chunks_by_split = {}
embeddings_by_split = {}

for split, (chunk_file, embedding_file) in september_files.items():
    chunks = pd.read_parquet(SEPT_DIR / chunk_file)
    saved = np.load(SEPT_DIR / embedding_file, allow_pickle=True)

    # the embeddings must be in the same order as the chunks, and the chunks in the same order as the Task 3 table
    assert (saved["chunk_ids"].astype(str) == chunks["chunk_id"].astype(str).to_numpy()).all()
    assert (chunks["chunk_id"].to_numpy() == task3_targets[split]["chunk_id"].to_numpy()).all()

    chunks_by_split[split] = chunks.reset_index(drop=True)
    embeddings_by_split[split] = saved["embeddings"]
    print(split, "chunks:", len(chunks), "| embeddings:", saved["embeddings"].shape)

train chunks: 10841 | embeddings: (10841, 768)
validation chunks: 2873 | embeddings: (2873, 768)


# Steps 3-5 continued: baseline predictions, checks and handoff

The cells below pick up from Step 3. Two things to know before running them:

1. The two embedding cells above are not needed for what follows. The September embeddings in `task12b_outputs/` were made with the same model (`all-distilroberta-v1`) for the same chunks as the October Task 3 tables, so we reuse them (and check below that the chunk IDs line up). Re-embedding would give the same numbers and take a long time. Note that `create_embeddings(train_targets, "validation")` above passes the *training* table, so the `october_task7_outputs/embeddings/october_validation_embeddings.npz` it writes would be wrong. Do not use it. The Task 3 targets also have no `chunk_text` column, so that call would fail anyway.
2. Task 3 calls its two splits `train` and `validation`. In `task12b_outputs/` the validation chunks are stored in `test_chunks.parquet` / `test_embeddings.npz` (an older name). These are the same 2,873 chunk IDs as `task3_labels_mask_validation.parquet`, which we confirm below.

## Step 4.1 Logistic regression baseline predictions

The frozen model gives a probability for each chunk and category. We keep the probability as the score and apply the frozen threshold stored in the model (`clf.threshold_`, 0.20, the same value Task 13 used) to get the 0/1 prediction.

In [39]:
# Make sure the model's category order matches the order of the labels/mask lists in the Task 3 tables
CATEGORY_IDS = [
    "governing_law", "renewal_term", "revenue_profit_sharing", "cap_on_liability", "uncapped_liability",
    "termination_for_convenience", "anti_assignment", "audit_rights", "license_grant", "exclusivity",
]
assert list(clf.categories) == CATEGORY_IDS

LR_THRESHOLD = clf.threshold_
assert LR_THRESHOLD == 0.20
print("Frozen logistic regression threshold:", LR_THRESHOLD)

lr_scores = {}
for split in ["train", "validation"]:
    chunk_ids = chunks_by_split[split]["chunk_id"]
    lr_scores[split] = clf.predict_proba(embeddings_by_split[split], index=chunk_ids)

lr_scores["validation"].head()

Frozen logistic regression threshold: 0.2


,governing_law,renewal_term,revenue_profit_sharing,cap_on_liability,uncapped_liability,termination_for_convenience,anti_assignment,audit_rights,license_grant,exclusivity
chunk_id,,,,,,,,,,
contract_0013__chunk_0000,0.008757,0.000427,0.005870,0.001239,0.000195,0.003376,0.002482,0.015619,0.000326,0.001548
contract_0013__chunk_0001,0.000203,0.000087,0.041565,0.000167,0.000041,0.000417,0.002490,0.060035,0.000231,0.002592
contract_0013__chunk_0002,0.000141,0.001534,0.028126,0.001739,0.000567,0.001774,0.000776,0.011911,0.004770,0.005226
contract_0013__chunk_0003,0.000705,0.000868,0.021732,0.000268,0.000090,0.001221,0.001258,0.183014,0.000385,0.001842
contract_0013__chunk_0004,0.002158,0.000826,0.001046,0.004438,0.000219,0.009023,0.005049,0.006320,0.000112,0.000690


## Step 4.2: Keyword baseline predictions

Same idea for the keyword baseline: we use the frozen scorecards and the thresholds that were selected in Task 11. The keyword scorer uses category names (e.g. "Anti-Assignment"), so we map them to the category IDs used everywhere else.

NOTE: This cell is REALLY slow (about 10 minutes) it scans every chunk for every phrase

The saved scorecards were pickled from inside the Task 11 notebook, so Python needs the same class definitions (`Scorecard`, `EvidenceGroup`, `Rule`, `EvidenceEffect`) in this notebook before it can load them. These are copied unchanged from `task11_keyword_baseline.ipynb`.

In [40]:
from enum import Enum
from dataclasses import dataclass

# copied from task11_keyword_baseline.ipynb (needed to unpickle frozen_scorecards.pkl)
class EvidenceEffect(Enum):
    ANCHOR = 2
    SUPPORT = 1
    COUNTER = -1
    VETO = 0

@dataclass
class Rule:
    rule_id: str
    phrases: list[str]
    source: str
    rationale: str
    reviewer: str
    behavior: str

@dataclass
class EvidenceGroup:
    group_id: str
    effect: EvidenceEffect
    rules: list[Rule]

@dataclass
class Scorecard:
    category: str
    anchor_groups: list[EvidenceGroup]
    supporting_groups: list[EvidenceGroup]
    counterevidence_groups: list[EvidenceGroup]
    veto_groups: list[EvidenceGroup]
    threshold: int = 1

In [41]:
from keyword_classifier import KeywordClassifier

with open("task11_outputs/frozen_scorecards.pkl", "rb") as file:
    frozen_scorecards = pickle.load(file)

# category name used by the keyword scorer -> category_id used in the Task 3 tables
# (same order as CATEGORY_IDS)
keyword_name_to_id = dict(zip(
    ["Governing Law", "Renewal Term", "Revenue/Profit Sharing", "Cap On Liability", "Uncapped Liability",
     "Termination For Convenience", "Anti-Assignment", "Audit Rights", "License Grant", "Exclusivity"],
    CATEGORY_IDS,
))
assert set(frozen_scorecards.keys()) == set(keyword_name_to_id.keys())

# frozen thresholds from Task 11 (selected on the September validation data)
keyword_thresholds = pd.read_csv("task11_outputs/selected_thresholds_v1.csv", index_col=0)
print(keyword_thresholds[["category_name", "threshold"]])

kc = KeywordClassifier(frozen_scorecards)

keyword_results = {}
for split in ["train", "validation"]:
    scores = kc.score(chunks_by_split[split])
    scored = kc.predict(scores, keyword_thresholds)
    scored["category_id"] = scored["category_name"].map(keyword_name_to_id)
    keyword_results[split] = scored
    print(split, "keyword rows:", len(scored))

                 category_name  threshold
0              Anti-Assignment          1
1                 Audit Rights          1
2             Cap On Liability          1
3                  Exclusivity          2
4                Governing Law          1
5                License Grant          1
6                 Renewal Term          1
7       Revenue/Profit Sharing          2
8  Termination For Convenience          3
9           Uncapped Liability          1
train keyword rows: 108410
validation keyword rows: 28730


## Step 5: Build the row-level table

One row per chunk and category (so 10 rows per chunk), for each split.

We attach the Task 3 label and mask to every row so that future rows can score the baselines and the October model in exactly the same way (masked rows are left out of the metrics).

In [47]:
def build_rows(split):
    targets = task3_targets[split].reset_index(drop=True)
    chunk_ids = targets["chunk_id"].to_numpy()
    labels = np.stack(targets["labels"].to_numpy())   # (n_chunks, 10)
    mask = np.stack(targets["mask"].to_numpy())       # (n_chunks, 10)

    # long format: one row per chunk-category pair
    rows = pd.DataFrame({
        "split": split,
        "chunk_id": np.repeat(chunk_ids, len(CATEGORY_IDS)),
        "category_id": np.tile(CATEGORY_IDS, len(chunk_ids)),
        "label": labels.ravel(),
        "mask": mask.ravel(),
    })

    # logistic regression score + prediction
    lr = lr_scores[split].loc[chunk_ids, CATEGORY_IDS].to_numpy()
    rows["lr_score"] = lr.ravel()
    rows["lr_prediction"] = (rows["lr_score"] >= LR_THRESHOLD).astype(int)

    # keyword score + prediction (merge on the pair, so row order can't get mixed up)
    kw = keyword_results[split][["chunk_id", "category_id", "score", "prediction"]]
    kw = kw.rename(columns={"score": "keyword_score", "prediction": "keyword_prediction"})
    rows = rows.merge(kw, on=["chunk_id", "category_id"], how="left", validate="one_to_one")
    return rows

baseline_rows = pd.concat([build_rows("train"), build_rows("validation")], ignore_index=True)

print(baseline_rows.shape)
baseline_rows.head(12)

(137140, 9)


,split,chunk_id,category_id,label,mask,lr_score,lr_prediction,keyword_score,keyword_prediction
0,train,contract_0001__chunk_0000,governing_law,0,1,0.007397,0,0,0
1,train,contract_0001__chunk_0000,renewal_term,0,1,0.000424,0,0,0
2,train,contract_0001__chunk_0000,revenue_profit_sharing,0,1,0.002433,0,0,0
3,train,contract_0001__chunk_0000,cap_on_liability,0,1,0.002147,0,0,0
4,train,contract_0001__chunk_0000,uncapped_liability,0,1,0.000054,0,0,0
5,train,contract_0001__chunk_0000,termination_for_convenience,0,1,0.000512,0,0,0
6,train,contract_0001__chunk_0000,anti_assignment,0,1,0.004431,0,0,0
7,train,contract_0001__chunk_0000,audit_rights,0,1,0.017547,0,0,0
8,train,contract_0001__chunk_0000,license_grant,0,1,0.102494,0,0,0
9,train,contract_0001__chunk_0000,exclusivity,1,1,0.391936,1,0,0


## Step 6: Validate the outputs

In [48]:
# 1. expected size: 10 categories for every chunk
assert len(baseline_rows) == (10841 + 2873) * 10
print("rows by split:\n", baseline_rows["split"].value_counts(), "\n")

# 2. exactly one row per split/chunk/category and no gaps
assert not baseline_rows.duplicated(["split", "chunk_id", "category_id"]).any()
assert baseline_rows.isna().sum().sum() == 0

# 3. every chunk has all 10 categories
assert (baseline_rows.groupby(["split", "chunk_id"]).size() == 10).all()

# 4. predictions are 0/1, probabilities are between 0 and 1
for col in ["label", "mask", "lr_prediction", "keyword_prediction"]:
    assert baseline_rows[col].isin([0, 1]).all(), col
assert baseline_rows["lr_score"].between(0, 1).all()

# 5. no chunk is shared between train and validation (no leakage)
train_ids = set(baseline_rows.loc[baseline_rows["split"] == "train", "chunk_id"])
validation_ids = set(baseline_rows.loc[baseline_rows["split"] == "validation", "chunk_id"])
assert train_ids.isdisjoint(validation_ids)

# 6. labels and mask match the Task 3 target table (its label is blank for masked pairs, so compare labels on unmasked pairs only)
check = pd.read_csv(TASK3_DIR / "task3_chunk_category_targets.csv")
check = check.merge(baseline_rows, on=["split", "chunk_id", "category_id"], suffixes=("_csv", ""), validate="one_to_one")
assert len(check) == len(baseline_rows)
assert (check["mask"] == (~check["is_masked"]).astype(int)).all()
unmasked = check[check["mask"] == 1]
assert (unmasked["label_csv"] == unmasked["label"]).all()

print("All checks passed")

rows by split:
 split
train         108410
validation     28730
Name: count, dtype: int64 

All checks passed


## Step 7: Save the outputs for Task 8

In [49]:
OUTPUT_DIR = Path("october_task7_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

row_level_path = OUTPUT_DIR / "task7_baseline_row_level_predictions.parquet"
baseline_rows.to_parquet(row_level_path, index=False)

# the thresholds used, so future tasks can see exactly what was applied
thresholds_used = pd.DataFrame({
    "baseline": ["logistic_regression"] * len(CATEGORY_IDS) + ["keyword"] * len(CATEGORY_IDS),
    "category_id": CATEGORY_IDS * 2,
    "threshold": [LR_THRESHOLD] * len(CATEGORY_IDS)
                 + [keyword_thresholds.set_index("category_name").loc[name, "threshold"] for name in keyword_name_to_id],
})
thresholds_path = OUTPUT_DIR / "task7_baseline_thresholds_used.csv"
thresholds_used.to_csv(thresholds_path, index=False)

print("Saved:", row_level_path, "|", len(baseline_rows), "rows")
print("Saved:", thresholds_path)

Saved: october_task7_outputs/task7_baseline_row_level_predictions.parquet | 137140 rows
Saved: october_task7_outputs/task7_baseline_thresholds_used.csv


## Last Note

`october_task7_outputs/task7_baseline_row_level_predictions.parquet` has one row per chunk and category for the train and validation splits (13,714 chunks x 10 categories = 137,140 rows):

| column | meaning |
|---|---|
| `split` | `train` or `validation` (Task 3 split) |
| `chunk_id`, `category_id` | the chunk-category pair |
| `label` | Task 3 target (1 = chunk fully contains a span of this category) |
| `mask` | Task 3 mask (1 = usable, 0 = masked, leave out of metrics) |
| `lr_score`, `lr_prediction` | September DistilRoBERTa embedding + logistic regression probability, and 0/1 at the frozen threshold (0.20) |
| `keyword_score`, `keyword_prediction` | September keyword score, and 0/1 at the frozen Task 11 thresholds |

Nothing was re-trained or re-tuned for this task.